# Assistente Tecnico Databricks - RAG com Embeddings Semanticos e ChromaDB

Este projeto implementa um assistente especializado em Engenharia de Dados e ecossistema Databricks, utilizando a arquitetura RAG (Retrieval-Augmented Generation) avancada.

Diferente de abordagens baseadas em palavras-chave (como TF-IDF), este sistema utiliza busca semantica densa: o significado da pergunta e comparado ao significado dos documentos atraves de vetores de alta dimensao, permitindo encontrar respostas mesmo quando o usuario usa termos diferentes dos textos originais.

### Componentes da Solucao:
- **Base de Conhecimento:** Documentacao tecnica oficial de Databricks (Medallion Architecture, Delta Lake, Unity Catalog, Delta Live Tables e Otimizacao com Photon).
- **Modelo de Embeddings:** Google `text-embedding-004` via SDK oficial `google-genai`.
- **Banco Vetorial:** `ChromaDB` em modo persistente local.
- **Modelo de Linguagem (LLM):** Google `gemini-3.5-flash-lite` com prompt ancorado no contexto recuperado.
- **Interface de Usuario:** `Gradio` para interacao via navegador.

## 1. Instalacao das Dependencias
Instalamos o SDK do Google GenAI, o ChromaDB e o Gradio.

In [ ]:
!pip install -q -U google-genai chromadb gradio requests

## 2. Autenticacao com a API do Google Gemini
Recuperacao segura da chave via Colab Secrets (`userdata.get`) ou variavel de ambiente.

In [ ]:
import os
import json
from google import genai

api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except (ImportError, Exception):
    api_key = os.environ.get("GEMINI_API_KEY")

# Tratamento se a chave tiver sido salva em formato de dicionario ou JSON
if isinstance(api_key, dict):
    for k in ["GEMINI_API_KEY", "api_key", "key", "token"]:
        if k in api_key and isinstance(api_key[k], str):
            api_key = api_key[k]
            break
    else:
        for v in api_key.values():
            if isinstance(v, str):
                api_key = v
                break

if isinstance(api_key, str):
    api_key = api_key.strip()
    if api_key.startswith("{") and api_key.endswith("}"):
        try:
            d = json.loads(api_key)
            if isinstance(d, dict):
                for k in ["GEMINI_API_KEY", "api_key", "key"]:
                    if k in d and isinstance(d[k], str):
                        api_key = d[k].strip()
                        break
        except Exception:
            pass

if not api_key or not isinstance(api_key, str):
    import getpass
    api_key = getpass.getpass("Digite sua GEMINI_API_KEY: ").strip()

client = genai.Client(api_key=api_key)
print("Cliente Gemini configurado com sucesso.")

## 3. Base de Conhecimento Oficial da Databricks
Carregamos os documentos tecnicos de referencia sobre os principais pilares do Databricks.
Se os arquivos locais nao estiverem presentes (por exemplo, ao executar em uma maquina limpa do Colab), o codigo baixa automaticamente os documentos do repositorio GitHub oficial do projeto.

In [ ]:
import os
import requests

DOCS_DIR = "data/databricks_docs"
os.makedirs(DOCS_DIR, exist_ok=True)

ARQUIVOS_BASE = [
    "medallion_architecture.md",
    "delta_lake_internals.md",
    "unity_catalog_governance.md",
    "delta_live_tables.md",
    "spark_optimization_photon.md"
]

BASE_GITHUB_URL = "https://raw.githubusercontent.com/miguelmorandini/chat-com-pdf/main/data/databricks_docs/"

documentos_carregados = []

for nome_arquivo in ARQUIVOS_BASE:
    caminho_local = os.path.join(DOCS_DIR, nome_arquivo)
    
    # Se nao existir localmente, faz download automatico do GitHub
    if not os.path.exists(caminho_local):
        try:
            url = BASE_GITHUB_URL + nome_arquivo
            resp = requests.get(url, timeout=10)
            if resp.status_code == 200:
                with open(caminho_local, "w", encoding="utf-8") as f:
                    f.write(resp.text)
        except Exception as e:
            print(f"Aviso: nao foi possivel baixar {nome_arquivo}: {e}")
    
    if os.path.exists(caminho_local):
        with open(caminho_local, "r", encoding="utf-8") as f:
            conteudo = f.read()
            documentos_carregados.append({
                "arquivo": nome_arquivo,
                "conteudo": conteudo
            })

print(f"Total de documentos carregados: {len(documentos_carregados)}")
for doc in documentos_carregados:
    print(f"- {doc['arquivo']} ({len(doc['conteudo'])} caracteres)")

## 4. Chunking Semantico com Metadados
Dividimos os documentos por secoes tecnicas (titulos em Markdown `##`), preservando o contexto e associando metadados como nome do arquivo e titulo da secao em cada bloco.

In [ ]:
def dividir_em_chunks_estruturados(documentos, tamanho_maximo=900, overlap=120):
    """
    Quebra documentos em blocos contextuais mantendo titulo e metadados.
    """
    chunks = []
    
    for doc in documentos:
        arquivo = doc["arquivo"]
        linhas = doc["conteudo"].split("\n")
        
        secao_atual = "Introducao"
        texto_secao = []
        
        for linha in linhas:
            if linha.startswith("## "):
                # Processa a secao anterior
                corpo = " ".join(" ".join(texto_secao).split())
                if corpo:
                    # Divide se exceder tamanho maximo
                    passo = tamanho_maximo - overlap
                    for inicio in range(0, max(1, len(corpo)), passo):
                        trecho = corpo[inicio:inicio + tamanho_maximo]
                        if len(trecho.strip()) > 40:
                            chunks.append({
                                "documento": arquivo,
                                "secao": secao_atual,
                                "texto": trecho.strip()
                            })
                secao_atual = linha.replace("## ", "").strip()
                texto_secao = []
            else:
                texto_secao.append(linha)
        
        # Ultima secao do documento
        corpo = " ".join(" ".join(texto_secao).split())
        if corpo:
            passo = tamanho_maximo - overlap
            for inicio in range(0, max(1, len(corpo)), passo):
                trecho = corpo[inicio:inicio + tamanho_maximo]
                if len(trecho.strip()) > 40:
                    chunks.append({
                        "documento": arquivo,
                        "secao": secao_atual,
                        "texto": trecho.strip()
                    })
                    
    return chunks

chunks = dividir_em_chunks_estruturados(documentos_carregados)
print(f"Total de chunks gerados com metadados: {len(chunks)}")
if chunks:
    print(f"Exemplo de chunk:\nDocumento: {chunks[0]['documento']}\nSecao: {chunks[0]['secao']}\nTexto: {chunks[0]['texto'][:180]}...")

## 5. Embeddings e Banco Vetorial com ChromaDB
Utilizamos o modelo Google `text-embedding-004` para converter cada trecho de texto em um vetor semantico de 768 dimensoes.
Os vetores e metadados sao gravados no `ChromaDB` em modo persistente (`./chroma_db`), garantindo que novas consultas nao precisem recalcular os embeddings.

In [ ]:
import chromadb
import time

# Inicializa o cliente do ChromaDB com armazenamento local
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection(
    name="databricks_knowledge_base",
    metadata={"hnsw:space": "cosine"}
)

def obter_embedding(texto):
    """Gera o vetor semantico usando o modelo text-embedding-004 da Google."""
    resp = client.models.embed_content(
        model="text-embedding-004",
        contents=texto
    )
    return resp.embeddings[0].values

# Indexa os chunks caso a colecao ainda nao esteja populada
total_existente = collection.count()
if total_existente < len(chunks):
    print(f"Indexando {len(chunks)} chunks no ChromaDB com text-embedding-004...")
    
    ids = []
    textos = []
    metadatas = []
    vetores = []
    
    for idx, c in enumerate(chunks):
        chunk_id = f"{c['documento']}_{idx}"
        texto_completo = f"Documento: {c['documento']} | Secao: {c['secao']}\n{c['texto']}"
        
        # Gera embedding
        vetor = obter_embedding(texto_completo)
        
        ids.append(chunk_id)
        textos.append(c["texto"])
        metadatas.append({
            "documento": c["documento"],
            "secao": c["secao"]
        })
        vetores.append(vetor)
        
        # Pausa leve para respeitar a taxa da API gratuita
        time.sleep(0.1)
    
    collection.add(
        ids=ids,
        documents=textos,
        metadatas=metadatas,
        embeddings=vetores
    )
    print(f"Indexacao concluida com sucesso! Total no banco: {collection.count()} registros.")
else:
    print(f"Banco vetorial ja indexado ({total_existente} chunks disponiveis no ChromaDB).")

## 6. Funcao de Busca Semantica no ChromaDB
Calcula a proximidade angular (cosseno) entre a intencao da pergunta e os trechos indexados.

In [ ]:
def buscar_documentos(pergunta, top_k=4):
    """
    Busca semantica de trechos relevantes no ChromaDB.
    """
    vetor_pergunta = obter_embedding(pergunta)
    
    res = collection.query(
        query_embeddings=[vetor_pergunta],
        n_results=top_k
    )
    
    resultados = []
    if res["documents"] and len(res["documents"][0]) > 0:
        for i in range(len(res["documents"][0])):
            resultados.append({
                "texto": res["documents"][0][i],
                "documento": res["metadatas"][0][i]["documento"],
                "secao": res["metadatas"][0][i]["secao"],
                "distancia": res["distances"][0][i] if "distances" in res else None
            })
            
    return resultados

print("Funcao de busca semantica configurada.")

## 7. Pipeline RAG com Gemini 3.5 Flash-Lite
Integramos a recuperacao semantica a sintese do modelo generativo. O prompt adota a persona de Especialista em Engenharia de Dados da Databricks, com regras restritas para citar fontes e recusar alucinacoes.

In [ ]:
def perguntar_databricks(pergunta, modelo="gemini-3.5-flash-lite", top_k=4):
    """
    Executa o ciclo completo de RAG:
    1. Busca trechos semanticamente relevantes no ChromaDB.
    2. Monta o contexto com informacoes de documento e secao.
    3. Invoca o Gemini com instrucao de completude e rigor anti-alucinacao.
    """
    trechos = buscar_documentos(pergunta, top_k=top_k)
    
    if not trechos:
        return "Nao encontrei informacoes sobre esse topico na documentacao fornecida."
    
    contexto = "\n\n".join([
        f"[Fonte: {t['documento']} | Secao: {t['secao']}]\n{t['texto']}"
        for t in trechos
    ])
    
    prompt = f"""
Voce e um especialista em Engenharia de Dados focado no ecossistema Databricks.
Responda a pergunta exclusivamente com base nas informacoes do CONTEXTO abaixo.

Regras obrigatorias:
1. Baseie-se apenas nas informacoes presentes no CONTEXTO.
2. Nao invente e nao presuma informacoes nao mencionadas.
3. Cite explicitamente a fonte e a secao utilizada para cada afirmacao importante.
4. Responda em portugues de forma estruturada, tecnica e detalhada.
5. Se a resposta nao estiver no documento, responda estritamente:
   "Nao encontrei essa informacao na documentacao disponivel."

CONTEXTO:
{contexto}

PERGUNTA:
{pergunta}
"""
    
    resposta = client.models.generate_content(
        model=modelo,
        contents=prompt
    )
    
    return resposta.text.strip()

print("Pipeline RAG do assistente Databricks configurado.")

## 8. Bateria de Testes de Engenharia de Dados
Validacao com perguntas reais sobre arquitetura, governanca, otimizacao e um teste anti-alucinacao.

In [ ]:
testes = [
    {
        "titulo": "Pergunta 1 - Arquitetura Medalhao",
        "pergunta": "Qual e a responsabilidade de cada camada (Bronze, Silver e Gold) na arquitetura Medalhao do Databricks?"
    },
    {
        "titulo": "Pergunta 2 - Funcionamento Interno do Delta Lake",
        "pergunta": "Como o log de transacoes (_delta_log) e os checkpoints garantem propriedades ACID no Delta Lake?"
    },
    {
        "titulo": "Pergunta 3 - Governanca com Unity Catalog",
        "pergunta": "Como funciona o namespace de tres niveis no Unity Catalog e quais as vantagens sobre o Hive Metastore?"
    },
    {
        "titulo": "Pergunta 4 - Teste Anti-Alucinacao (Fora do Escopo)",
        "pergunta": "Quem foi o campeao da Copa Libertadores da America em 2019?"
    }
]

print("=" * 70)
print("EXECUCAO DOS TESTES DO ASSISTENTE DATABRICKS")
print("=" * 70)

for t in testes:
    print(f"\n[{t['titulo']}]")
    print(f"Pergunta: {t['pergunta']}")
    resp = perguntar_databricks(t["pergunta"])
    print(f"\nResposta:\n{resp}")
    print("-" * 70)

## 9. Interface Interativa com Gradio
Interface web para tirar duvidas em tempo real sobre Databricks e Engenharia de Dados.

In [ ]:
import gradio as gr

def responder_chat(mensagem, historico):
    if not mensagem.strip():
        return "Por favor, digite uma pergunta valida."
    return perguntar_databricks(mensagem)

demo = gr.ChatInterface(
    fn=responder_chat,
    title="Assistente Tecnico Databricks - RAG Especialista",
    description=(
        "Tire duvidas tecnicas sobre arquitetura Medalhao, Delta Lake, Unity Catalog, "
        "Delta Live Tables e otimizacao com Photon. "
        "Respostas geradas com base estrita na documentacao oficial com citacao de fontes."
    ),
    examples=[
        "Qual a diferenca entre a camada Bronze e Silver?",
        "Como o Delta Lake lida com transacoes concorrentes?",
        "O que e o namespace de tres niveis no Unity Catalog?",
        "Quais as vantagens do motor Photon sobre a JVM tradicional?"
    ]
)

demo.launch(share=True, debug=False)